# WCC simulated image of a globular cluster

Uses `wcc_sim.simulate_field` to render a photorealistic WCC detector image of the M13
globular cluster core (NGC 6205, RA 250.42°, Dec +36.46°) from the Gaia DR3 star catalogue.
The Gaia query is cached on disk so re-runs are fast.

For a time-series / crowded-field photometry analysis see
`lazuli/science_gallery/11_crowded_field_photometry.ipynb` (M4 core).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from wcc_sim import simulate_field
from wcc_sim.catalog import query_gaia

# M13 (NGC 6205) — bright northern globular, ~145 arcsec core radius
RA, DEC = 250.4235, 36.4613
CLUSTER  = "M13 (NGC 6205)"
CACHE    = "gaia_cache"          # Gaia query cached here on first run

In [ ]:
# Fetch Gaia DR3 sources and simulate one WCC exposure.
# shape=(2048,2048) is a 34" subarray centred on the core; omit for the full 9576x6388 frame.
# First run takes ~60 s (Gaia query + rendering); subsequent runs use the cache.
cat = query_gaia(RA, DEC, radius_arcsec=60, mag_limit=21, cache_dir=CACHE)

field = simulate_field(
    RA, DEC,
    sensorfilter="zwo:r",
    focus=1,          # 1-wave defocus — spreads PSF for precision aperture photometry
    exptime=30,       # short exposure keeps the core unsaturated
    shape=(2048, 2048),
    seed=42,
    catalog=cat,
    cache_dir=CACHE,
)

PLATE = field.params["plate_scale_mas"]
fov_arcsec = field.image_e.shape[0] * PLATE / 1000
print(f"{len(cat)} Gaia sources queried")
print(f"Image: {field.image_e.shape}, {PLATE:.2f} mas/px, {fov_arcsec:.1f}\" on a side")
print(f"Params: exptime={field.params['exptime']:.0f}s, "
      f"read_noise={field.params['read_noise']:.1f} e-, "
      f"sky={field.params['sky_e_s']:.4f} e-/s/px")

In [ ]:
# Quick-look image with the Gaia truth catalogue overlaid
ax = field.plot(catalog=True)
ax.set_title(f"{CLUSTER} — zwo:r, focus=1, {field.params['exptime']:.0f} s, {fov_arcsec:.0f}\" field")
plt.show()

In [ ]:
# Star census
c = field.catalog
in_image  = np.asarray(c["in_image"],  bool)
saturated = np.asarray(c["saturated"], bool)
mags = np.asarray(c["phot_g_mean_mag"], float)

print(f"{in_image.sum()} stars in the frame")
print(f"  saturated : {(in_image & saturated).sum()}  "
      f"(brightest G = {mags[in_image].min():.1f})")
print(f"  measurable: {(in_image & ~saturated).sum()}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(mags[in_image & ~saturated], bins=np.arange(13, 22, 0.5),
        color="steelblue", edgecolor="white", linewidth=0.4)
ax.set(xlabel="Gaia G [mag]", ylabel="stars per 0.5 mag bin",
       title=f"{CLUSTER}: unsaturated stars in the 34\" WCC subarray")
plt.tight_layout()
plt.show()

In [ ]:
# Save to FITS (SCI + SATMASK + CAT + CLEAN extensions)
# field.to_hdulist().writeto("m13_wcc_r_30s.fits", overwrite=True)
# print("Written m13_wcc_r_30s.fits")